<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Moyenne horaire - RMSE public : 4.8189
</div>
<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des bilbiothèques
</div>

In [11]:
import pandas as pd

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des documents de données temporelles et de test
</div>

In [12]:
history_toulouse_train = pd.read_parquet("history_toulouse_train.parquet")
history_toulouse_test = pd.read_parquet("history_toulouse_test.parquet")

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Prédictions
</div>
On convertit les indexe en colonnes classiques

In [13]:
train = history_toulouse_train.reset_index()

On convertit les chaines de caractère en format date en on extrait les heures

In [14]:
train["heure"] = train["commit_at"].dt.hour

Moyenne horaire de chaque station

In [15]:
moyenne_station_heure = (train.groupby(["station", "heure"])["bikes"].mean().round().astype(int))

Moyenne totale de la station en cas de valeurs manquantes

In [16]:
moyenne_station = train.groupby("station")["bikes"].mean().round().astype(int)

On extrait tous les indexes station et commit_at_index sous forme de liste

In [17]:
station_index = history_toulouse_test.index.get_level_values("station")
commit_at_index = history_toulouse_test.index.get_level_values("commit_at")

On associe chaque nom de colonne à une valeur numérique

In [18]:
decalage = {
    "bikes_+15min": 15,
    "bikes_+30min": 30,
    "bikes_+45min": 45,
    "bikes_+60min": 60,
    "bikes_+75min": 75,
    "bikes_+90min": 90,
    "bikes_+105min": 105,
    "bikes_+120min": 120,
}

On procède pour chaque décalage individuellement

In [19]:
for col, h in decalage.items():
    # Instant cible = commit_at + h minutes
    heure_cible = (commit_at_index + pd.Timedelta(minutes=h)).hour
    # On relie chaque station à son heure
    cle = list(zip(station_index, heure_cible))
    # On associe chaque (station_index, heure_cible) à la moyenne horaire du nombre de vélos
    valeurs = pd.Series(cle, index=history_toulouse_test.index).map(moyenne_station_heure)
    # En cas de valeur manquante, on utilise la moyenne totale de la station
    valeurs = valeurs.fillna(pd.Series(station_index, index=history_toulouse_test.index).map(moyenne_station))
    # On sauvegarde le résultat en tant qu'entier
    history_toulouse_test[col] = valeurs.astype(int)

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Sauvegarde du résultat
</div>

In [20]:
history_toulouse_test.to_parquet("BA_prediction_2_horaire.parquet", index=True)
history_toulouse_test

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,4,4,4,4,3,3,3,3
00002 - LAFAYETTE,2026-07-31 23:45:00,5,5,5,5,4,4,4,4
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,11,11,11,11,10,10,10,10
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,4,4,4,4,2,2,2,2
00003 - POMME,2025-01-05 05:45:00,8,8,8,8,9,9,9,9
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,0,0,0,0,0,0,0,0
PASTEUR - LAVOISIER,2025-08-05 10:15:00,0,0,0,0,0,0,0,0
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,0,0,0,0,0,0,0,0
